In [3]:
import os
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def fetch_and_save(url_sid_folder):
    url, sid, folder = url_sid_folder
    out_path = os.path.join(folder, f"{sid}.jpg")
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(resp.content)
    except:
        pass  

def fast_bulk_download(csv_path, folder, n_workers=16):
    df = pd.read_csv(csv_path)
    os.makedirs(folder, exist_ok=True)
    tasks = [(row["image_link"], row["sample_id"], folder) for idx, row in df.iterrows()]
    with ThreadPoolExecutor(max_workers=n_workers) as exe:
        list(tqdm(exe.map(fetch_and_save, tasks), total=len(tasks)))
    print(f"Download complete! Downloaded {len(tasks)} images to {folder}/")
fast_bulk_download('train.csv', 'images/train', n_workers=24)
fast_bulk_download('test.csv',  'images/test',  n_workers=24)


100%|██████████| 75000/75000 [20:46<00:00, 60.18it/s]  


Download complete! Downloaded 75000 images to images/train/


100%|██████████| 75000/75000 [17:44<00:00, 70.46it/s]  

Download complete! Downloaded 75000 images to images/test/


In [4]:
!pip install transformers torch torchvision scikit-learn pandas numpy tqdm --quiet

import os
import re
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from PIL import Image
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DATA LOADING
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train['image_path'] = train['sample_id'].apply(lambda x: f"images/train/{x}.jpg")
test['image_path'] = test['sample_id'].apply(lambda x: f"images/test/{x}.jpg")

# TEXT CLEANING
def clean_text(text):
    if pd.isna(text): return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.lower().strip()
train['clean_text'] = train['catalog_content'].apply(clean_text)
test['clean_text'] = test['catalog_content'].apply(clean_text)

# IPQ EXTRACTION
def extract_ipq(text):
    m = re.search(r"(\d+)\s*(pack|pcs|piece|pieces|count|qty)", text)
    return int(m.group(1)) if m else 1
train['ipq'] = train['catalog_content'].apply(extract_ipq)
test['ipq'] = test['catalog_content'].apply(extract_ipq)
scaler = StandardScaler()
train_ipq = scaler.fit_transform(train[['ipq']]).flatten()
test_ipq = scaler.transform(test[['ipq']]).flatten()

# LOG TRANSFORMED TARGET
train['price_log'] = np.log1p(train['price'])

# TOKENIZER
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# IMAGE TRANSFORM 
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# MULTIMODAL DATASET
class PriceDataset(Dataset):
    def __init__(self, texts, ipqs, img_paths, targets=None, max_len=256):
        self.texts = texts
        self.ipqs = ipqs
        self.img_paths = img_paths
        self.targets = targets
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Text
        enc = tokenizer(self.texts[idx], max_length=self.max_len, padding='max_length',
                        truncation=True, return_tensors='pt')
        item = {k:v.squeeze(0) for k,v in enc.items()}
        item['ipq'] = torch.tensor(self.ipqs[idx], dtype=torch.float)

        # Image
        img_path = self.img_paths[idx]
        try:
            img = Image.open(img_path).convert("RGB")
            img = img_transform(img)
        except Exception:  # fallback if image missing/corrupted
            img = torch.zeros(3, 224, 224)
        item['image'] = img

        # Label
        if self.targets is not None:
            item['labels'] = torch.tensor(self.targets[idx], dtype=torch.float)
        return item

# EfficientNet + DistilBert + IPQ
class MultiModalRegressor(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.bert_dropout = nn.Dropout(dropout)
        self.img_encoder = models.efficientnet_b0(pretrained=True)
        for param in self.img_encoder.parameters(): param.requires_grad = False
        self.img_fc = nn.Linear(1280, 128)
        self.fusion_fc = nn.Linear(self.bert.config.hidden_size + 128 + 1, 64)
        self.final_fc = nn.Linear(64, 1)
        self.final_dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask, ipq, images):
        # Text features
        text_feat = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0,:]
        text_feat = self.bert_dropout(text_feat)
        # Image features
        img_feat = self.img_encoder.features(images)
        img_feat = nn.functional.adaptive_avg_pool2d(img_feat, (1,1)).reshape(images.size(0), -1)
        img_feat = self.img_fc(img_feat)
        # IPQ
        ipq = ipq.unsqueeze(1)
        # Fusion
        x = torch.cat([text_feat, img_feat, ipq], dim=1)
        x = torch.relu(self.fusion_fc(x))
        x = self.final_dropout(x)
        return self.final_fc(x).squeeze(1)

# TRAIN/VAL SPLIT
X_train, X_val, ipq_train, ipq_val, img_train, img_val, y_train, y_val = train_test_split(
    train['clean_text'].values, train_ipq, train['image_path'].values,
    train['price_log'].values.astype(np.float32),
    test_size=0.1, random_state=42
)
train_dataset = PriceDataset(X_train, ipq_train, img_train, y_train)
val_dataset = PriceDataset(X_val, ipq_val, img_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=4)

# TRAINING
model = MultiModalRegressor().to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5)
loss_fn = nn.MSELoss()
epochs = 10

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return 100*np.mean(2*np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ipq = batch['ipq'].to(device)
        images = batch['image'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask, ipq, images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
    print(f"Epoch {epoch+1} Avg Train Loss: {train_loss/len(train_loader.dataset):.4f}")

    # Validation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Val"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ipq = batch['ipq'].to(device)
            images = batch['image'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask, ipq, images)
            val_preds.extend(np.expm1(outputs.cpu().numpy()))
            val_targets.extend(np.expm1(labels.cpu().numpy()))
    print(f"Epoch {epoch+1} Validation SMAPE: {smape(val_targets, val_preds):.4f}")

# INFERENCE
test_dataset = PriceDataset(
    test['clean_text'].values, test_ipq, test['image_path'].values
)
test_loader = DataLoader(test_dataset, batch_size=16, num_workers=4)
model.eval()
test_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Prediction"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ipq = batch['ipq'].to(device)
        images = batch['image'].to(device)
        outputs = model(input_ids, attention_mask, ipq, images)
        test_preds.extend(np.expm1(outputs.cpu().numpy()))

# OUTPUT SUBMISSION
submission = pd.DataFrame({'sample_id': test['sample_id'],
                           'price': np.clip(test_preds, 0.01, None)})
submission.to_csv("test_out.csv", index=False)
print("File saved as test_out.csv")


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1 Train: 100%|██████████| 4219/4219 [06:35<00:00, 10.67it/s]


Epoch 1 Avg Train Loss: 0.6884


Epoch 1 Val: 100%|██████████| 469/469 [00:40<00:00, 11.68it/s]


Epoch 1 Validation SMAPE: 52.3808


Epoch 2 Train: 100%|██████████| 4219/4219 [06:33<00:00, 10.73it/s]


...

Epoch 10 Avg Train Loss: 0.2144


Epoch 10 Val: 100%|██████████| 469/469 [00:40<00:00, 11.63it/s]


Epoch 10 Validation SMAPE: 47.2768


Test Prediction: 100%|██████████| 4688/4688 [06:42<00:00, 11.65it/s]


File saved as test_out.csv
